# 02 - Shot distribution and goal rate plots

**Goal:** answer two questions with plots:
1. *Where, when and how are shots taken?* (distribution)
2. *In which situations does a shot become a goal more often?* (goal rate)

**Decision: which shots do we plot?** We plot **open-play shots only** (`shot_type == "shot"`)
in the main analysis, and show penalties and free-kick shots **separately** (section 2).
The reasons are explained in section 2 with numbers.

Run the cells from top to bottom. Make sure the kernel is the **`.venv`** environment.

## 1. Setup and data
`eda_utils.py` (in `src/`) holds the helper functions we use here: confidence intervals,
a bar-chart function for goal rates, and a function that draws the pitch.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(str(Path("..").resolve() / "src"))
from eda_utils import draw_half_pitch, goal_rate_table, plot_goal_rate

FIG_DIR = Path("..") / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})


def save(fig, name):
    """Save a figure into reports/figures."""
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")


shots_all = pd.read_csv("../data/processed/shots_all.csv")
shots = shots_all[shots_all["shot_type"] == "shot"].copy()   # open-play shots only

print("All shot types:", len(shots_all), "| open play:", len(shots))
shots_all.head()

## 2. Decision: separate or together?
First we look at the three shot types next to each other.

- Left: how many shots of each type there are.
- Right: goal rate of each type. The black lines are **95% confidence intervals**
  (the range the true rate could plausibly be in; a wide line means few shots).

In [ ]:
by_type = goal_rate_table(shots_all, "shot_type")
display(by_type.round(3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(by_type.index, by_type["shots"], color="#2a6f97")
ax1.set_yscale("log")
ax1.set_title("Number of shots (log scale)")
ax1.set_ylabel("Shots")
plot_goal_rate(ax2, by_type, title="Goal rate by shot type")
fig.tight_layout()
save(fig, "01_shot_types.png")

Now we test what happens to the goal rate by distance if we **mix** all types together
versus using open play only. Distance here is `100 - start_x`: the distance to the goal line
in percent of the pitch length (10 is roughly 10.5 m). True distance and angle come on Day 3.

In [ ]:
bins = [0, 5, 10, 15, 20, 30, 100]
for df in (shots_all, shots):
    df["dist_to_goal_line"] = 100 - df["start_x"]
    df["dist_bin"] = pd.cut(df["dist_to_goal_line"], bins, right=False)

comparison = pd.DataFrame({
    "all types mixed": goal_rate_table(shots_all, "dist_bin")["goal_rate"],
    "open play only": goal_rate_table(shots, "dist_bin")["goal_rate"],
})
comparison["difference"] = comparison["all types mixed"] - comparison["open play only"]
comparison.round(3)

In [ ]:
# Where are penalties taken? (a tiny area, almost always the same spot)
shots_all[shots_all["shot_type"] == "penalty"][["start_x", "start_y"]].describe().round(1)

**Decision: separate.** Reasons:

1. **Penalties are not "normal" shots.** All 80 are taken from almost the same spot
   (x 89-91, y 46-51) and 70% are goals. In the 10-15 distance bin they push the goal rate
   from 10.9% (open play) to 12.7% (mixed). That is a wrong picture of how a shot from that
   distance usually ends.
2. **Free kicks distort the long-range shots.** In the 20-30 bin, 245 of 2,557 shots are
   free kicks, which raises the rate from 2.2% to 2.6%.
3. **They are different events.** A penalty has no defenders and the keeper must guess; a
   free kick is a set piece with a wall. A model trained on open play should not learn from
   them, and an xG model usually treats penalties as a fixed value.
4. **Nothing is lost.** The `shot_type` column lets us show them in their own plots
   (section 2 above and 3.2 below).

So: **main plots = open play only.** Penalties and free kicks get their own comparison.

## 3. Shot distribution (where and when)

### 3.1 Shot locations (open play)
A hexbin map counts shots in small hexagonal cells; darker means more shots.
The goal is on the right at x = 100.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
draw_half_pitch(ax)
hb = ax.hexbin(shots["start_x"], shots["start_y"], gridsize=28, extent=(50, 100, 0, 100),
               cmap="YlOrRd", mincnt=1, zorder=0)
fig.colorbar(hb, ax=ax, label="Shots", shrink=0.7)
ax.set_title("Open-play shot locations")
save(fig, "02_shot_map.png")

### 3.2 Penalties and free-kick shots (shown separately)
Each dot is one shot; green = goal, grey = no goal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, kind, title in zip(axes, ["penalty", "free_kick_shot"], ["Penalties", "Direct free-kick shots"]):
    sub = shots_all[shots_all["shot_type"] == kind]
    draw_half_pitch(ax)
    miss = sub[~sub["is_goal"]]
    goal = sub[sub["is_goal"]]
    ax.scatter(miss["start_x"], miss["start_y"], s=18, c="#999999", alpha=0.7, label="No goal")
    ax.scatter(goal["start_x"], goal["start_y"], s=18, c="#2a9d8f", alpha=0.9, label="Goal")
    ax.set_title(f"{title} (n={len(sub)})")
    ax.legend(loc="lower left", fontsize=8)
fig.tight_layout()
save(fig, "03_penalties_free_kicks.png")

### 3.3 Distance to the goal line
How far from the goal line are shots taken?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(shots["dist_to_goal_line"], bins=np.arange(0, 62, 2), color="#2a6f97", edgecolor="white")
ax.set_xlabel("Distance to goal line (% of pitch length)")
ax.set_ylabel("Shots")
ax.set_title("Open-play shots by distance")
save(fig, "04_distance_hist.png")

print("Share of shots closer than 20%:", round((shots["dist_to_goal_line"] < 20).mean(), 3))
print("Share of shots 30% or further :", round((shots["dist_to_goal_line"] >= 30).mean(), 3))

### 3.4 Shots over the course of each half
`eventSec` restarts at 0 in the second half, so we plot the two halves separately.
Minutes above 45 (first half) or about 45 (second half) are stoppage time.

In [ ]:
shots["minute"] = shots["eventSec"] / 60

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, half, name in zip(axes, ["1H", "2H"], ["First half", "Second half"]):
    sub = shots[shots["matchPeriod"] == half]
    ax.hist(sub["minute"], bins=np.arange(0, 61, 5), color="#2a6f97", edgecolor="white")
    ax.axvline(45, color="#aa3333", linestyle="--", linewidth=1)
    ax.set_title(f"{name} (n={len(sub)})")
    ax.set_xlabel("Minute within the half")
axes[0].set_ylabel("Shots")
fig.tight_layout()
save(fig, "05_shots_by_minute.png")

**Interpretation (distribution):**
- **Location:** shots are strongly concentrated in front of goal, inside and just outside the
  penalty area. About 70% of open-play shots are closer than 20% of the pitch length (about 21 m),
  and only about 3% are taken from 30% or further.
- **Penalties** all sit on the penalty spot; **free kicks** are taken from outside the penalty
  area, spread over a wide band roughly 15-35% (about 15-35 m) from the goal line.
- **Timing:** the second half has more shots (4,497 vs 3,954). Within a half the count rises
  slightly after the first 15 minutes. The shots above minute 45 (176 in the first half, 398 in the
  second) come from stoppage time.

## 4. Goal rate (open play)
For every bar: the **goal rate** is `goals / shots`. The black line is the 95% confidence interval
and `n` is the number of shots. Read the bars together with `n`; a high rate from a small group
is not reliable.

### 4.1 By distance to the goal line

In [ ]:
dist_table = goal_rate_table(shots, "dist_bin")
display(dist_table.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_goal_rate(ax, dist_table, title="Goal rate by distance to goal line (open play)",
               xlabel="Distance to goal line (% of pitch length)")
save(fig, "06_goal_rate_distance.png")

### 4.2 By how central the shot is
`|y - 50|` is how far the shot is from the centre line (0 = straight in front of goal).

In [ ]:
shots["offset_from_centre"] = (shots["start_y"] - 50).abs()
shots["centre_bin"] = pd.cut(shots["offset_from_centre"], [0, 5, 10, 20, 50], right=False)

centre_table = goal_rate_table(shots, "centre_bin")
display(centre_table.round(3))

fig, ax = plt.subplots(figsize=(7, 4.5))
plot_goal_rate(ax, centre_table, title="Goal rate by distance from the centre line (open play)",
               xlabel="|y - 50| (% of pitch width)")
save(fig, "07_goal_rate_centrality.png")

### 4.3 By body part

In [ ]:
shots["body_part"] = np.select(
    [shots["header"], shots["left_foot"], shots["right_foot"]],
    ["header", "left foot", "right foot"],
    default="other",
)
body_table = goal_rate_table(shots, "body_part")
display(body_table.round(3))

fig, ax = plt.subplots(figsize=(6, 4.5))
plot_goal_rate(ax, body_table, title="Goal rate by body part (open play)")
save(fig, "08_goal_rate_body_part.png")

In [ ]:
# Why do headers score more? Check how far from goal each body part shoots.
shots.groupby("body_part")["dist_to_goal_line"].median().round(1)

### 4.4 Counter attacks vs other attacks

In [ ]:
counter_table = goal_rate_table(shots, "counter_attack")
counter_table.index = counter_table.index.map({False: "other attacks", True: "counter attack"})
display(counter_table.round(3))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
plot_goal_rate(ax, counter_table, title="Goal rate: counter attacks (open play)")
save(fig, "09_goal_rate_counter.png")

**Interpretation (goal rate):**
- **Distance is the strongest pattern.** The goal rate falls steadily from 34.3% (closer than 5%)
  to 19.2%, 10.9%, 6.5% and 2.2% (20-30%). Beyond 30% it stays around 2-3% (only 263 shots, so the
  interval is wide: 1.3%-5.4%).
- **Centrality also matters:** 17.7% when the shot is within 5% of the centre line, down to 3.5%
  when it is 20% or more away from the centre (a very wide angle).
- **Headers score more often than foot shots (13.5% vs about 10%) - but be careful.**
  Headers are taken from much closer (median distance 8 vs 15). The difference is likely about
  *where* headers happen (crosses into the six-yard box), not about headers being better.
  This is a **confounding variable**; the model on Day 4 has to include distance to separate the two.
- **Left vs right foot** have almost the same goal rate (10.5% vs 10.2%).
- **Counter attacks** convert almost twice as often (19.8% vs 10.3%). The intervals do not
  overlap, so with 425 counter-attack shots this looks like a real difference. A counter attack
  often reaches the shooter against a disorganised defence, so it is a useful feature.

## 5. Takeaways for the next steps
- Use **open-play shots** for the model; handle **penalties** separately (fixed value, ~70% here) and
  decide later about free kicks.
- **Distance and centrality** are the key features. Day 3 computes real distance in metres and the
  shooting angle from the goal posts.
- Body part and counter attack are useful, but their effect overlaps with distance. Compare them
  in the Day 4 logistic regression, not just in these plots.
- All figures are saved in `reports/figures/`.